In [1]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "a9d2f1d5-517a-4ee4-bd7f-4e8dced3c8c9",
   "metadata": {},
   "outputs": [],
   "source": [
    "%load_ext autoreload\n",
    "%autoreload 2\n",
    "import uproot\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import awkward as ak\n",
    "import sys\n",
    "import importlib\n",
    "import coffea.util as util\n",
    "import time\n",
    "import json\n",
    "import os\n",
    "import numba as nb\n",
    "import awkward.numba\n",
    "\n",
    "sys.path.append(\"../analysisTools/\")\n",
    "from analysisTools import Analyzer\n",
    "from analysisTools import loadSchema\n",
    "import analysisTools as tools\n",
    "import analysisSubroutines as routines\n",
    "\n",
    "sample_cfg_dir = \"../configs/sample_configs/\"\n",
    "histo_cfg_dir = \"../configs/histo_configs/\"\n",
    "cut_cfg_dir = \"../configs/cut_configs/\""
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 2,
   "id": "04eff68a-bae6-44db-98d7-84f09ce85f8e",
   "metadata": {},
   "outputs": [],
   "source": [
    "# load in a \"sample config file\" for the 2018 signal samples\n",
    "with open(sample_cfg_dir+\"signal_v2_2018_aEM.json\",\"r\") as fin:\n",
    "    sig_cfg = json.load(fin)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 4,
   "id": "60367dc0-1d44-4bde-a5a4-f09d001168fc",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "{'location': '/store/group/lpcmetx/iDMe//Samples/Ntuples/signal_v2/2018/Mchi-22p0_dMchi-4p0/ctau-10/',\n",
       " 'Mchi': 22.0,\n",
       " 'dMchi': 4.0,\n",
       " 'ctau': 10,\n",
       " 'name': 'sig_Mchi-22.0_dMchi-4.0_ct-10',\n",
       " 'sum_wgt': 0.00014601059683627682,\n",
       " 'type': 'signal',\n",
       " 'year': 2018,\n",
       " 'alphaD': 'aEM',\n",
       " 'xsec': 99.19946999999999,\n",
       " 'nFiles': 8,\n",
       " 'num_events': 225012,\n",
       " 'blacklist': []}"
      ]
     },
     "execution_count": 4,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "# look at one of the entries\n",
    "sig_cfg[10]"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "4c29aae9-cf1d-49c9-abd3-5e10175adbcd",
   "metadata": {},
   "source": [
    "Here's what this information means:\n",
    "- `location` : the directory where all the `.root` files for this signal sample are located \n",
    "- `Mchi`, `dMchi`, `ctau` : model parameters for this signal point\n",
    "- `name` : name for the signal sample\n",
    "- `sum_wgt` : sum of the Monte Carlo event weights for all events in this sample (relevant for filling histograms)\n",
    "- `type`,`year` : Information on what kind of sample this is (signal, in this case), and what data-taking year the Monte Carlo generation was configured to reproduce\n",
    "- `alphaD` : parameter setting for the dark matter U(1) coupling constant. `aEM` means it's set equal to the electromagnetic coupling.\n",
    "- `xsec` : production cross section for this signal\n",
    "- `nFiles` : number of `.root` files in the sample\n",
    "- `num_events` : number of events in the sample\n",
    "- `blacklist` : list of any corrupted or unusable `.root` files (rare for this to happen)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "2df3ecb9-d34d-474e-8aef-4bc3c54bdc3a",
   "metadata": {},
   "source": [
    "Now, let's make a list of all the `.root` files for this sample. We will use the `XRootD` package to interact with the `eos` filesystem, where the `.root` files are stored."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 6,
   "id": "4fa175fa-6839-4149-b231-f1e8c9215996",
   "metadata": {},
   "outputs": [],
   "source": [
    "from XRootD import client\n",
    "loc = sig_cfg[10]['location']\n",
    "blacklist = sig_cfg[10]['blacklist']\n",
    "xrdClient = client.FileSystem(\"root://cmseos.fnal.gov\")\n",
    "status, flist = xrdClient.dirlist(loc) # get list of files in directory\n",
    "# select only non-blacklisted root files, prepend the root://cmsxrootd.fnal.gov/ to the file paths\n",
    "fullList = [\"root://cmseos.fnal.gov/\"+loc+\"/\"+item.name for item in flist if (('.root' in item.name) and (item.name not in blacklist))]"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 7,
   "id": "4c1df55c-f98e-4360-bda5-dd09802b7516",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "['root://cmseos.fnal.gov//store/group/lpcmetx/iDMe//Samples/Ntuples/signal_v2/2018/Mchi-22p0_dMchi-4p0/ctau-10//ntuples_Mchi-22p0_dMchi-4p0_ctau-10_00.root',\n",
       " 'root://cmseos.fnal.gov//store/group/lpcmetx/iDMe//Samples/Ntuples/signal_v2/2018/Mchi-22p0_dMchi-4p0/ctau-10//ntuples_Mchi-22p0_dMchi-4p0_ctau-10_01.root',\n",
       " 'root://cmseos.fnal.gov//store/group/lpcmetx/iDMe//Samples/Ntuples/signal_v2/2018/Mchi-22p0_dMchi-4p0/ctau-10//ntuples_Mchi-22p0_dMchi-4p0_ctau-10_02.root',\n",
       " 'root://cmseos.fnal.gov//store/group/lpcmetx/iDMe//Samples/Ntuples/signal_v2/2018/Mchi-22p0_dMchi-4p0/ctau-10//ntuples_Mchi-22p0_dMchi-4p0_ctau-10_03.root',\n",
       " 'root://cmseos.fnal.gov//store/group/lpcmetx/iDMe//Samples/Ntuples/signal_v2/2018/Mchi-22p0_dMchi-4p0/ctau-10//ntuples_Mchi-22p0_dMchi-4p0_ctau-10_04.root',\n",
       " 'root://cmseos.fnal.gov//store/group/lpcmetx/iDMe//Samples/Ntuples/signal_v2/2018/Mchi-22p0_dMchi-4p0/ctau-10//ntuples_Mchi-22p0_dMchi-4p0_ctau-10_05.root',\n",
       " 'root://cmseos.fnal.gov//store/group/lpcmetx/iDMe//Samples/Ntuples/signal_v2/2018/Mchi-22p0_dMchi-4p0/ctau-10//ntuples_Mchi-22p0_dMchi-4p0_ctau-10_06.root',\n",
       " 'root://cmseos.fnal.gov//store/group/lpcmetx/iDMe//Samples/Ntuples/signal_v2/2018/Mchi-22p0_dMchi-4p0/ctau-10//ntuples_Mchi-22p0_dMchi-4p0_ctau-10_07.root']"
      ]
     },
     "execution_count": 7,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "fullList"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "7832636e-c15c-494d-bbfd-173353c4ea29",
   "metadata": {
    "jp-MarkdownHeadingCollapsed": true,
    "tags": []
   },
   "source": [
    "## Using `uproot` to open files"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "b907f8a2-e5ac-497f-a76b-bf89da8255cc",
   "metadata": {},
   "source": [
    "Now we will use the uproot package to load in a ROOT file and look at its contents. All of the event data is stored in a `TTree` object called `outT` inside the `ntuples` directory of the file."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 16,
   "id": "08beef62-e052-41b3-9e4a-a9eac091de6a",
   "metadata": {},
   "outputs": [],
   "source": [
    "# open the first file in the list\n",
    "t = uproot.open(fullList[0])['ntuples/outT']"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 18,
   "id": "6809295e-bf6a-42f2-bb26-710b2c3321f6",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "<TTree 'outT' (296 branches) at 0x7f52330b9d90>"
      ]
     },
     "execution_count": 18,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "t"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "4b6e171f-e5b9-478d-b6db-cc5de5e06414",
   "metadata": {},
   "source": [
    "We see that the TTree has 296 \"branches\". Each branch records the value(s) of some variable for each event (e.g. electron $p_T$ or missing transverse energy)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 19,
   "id": "76ac6597-083d-4437-91a6-d5493a0f65d3",
   "metadata": {
    "collapsed": true,
    "jupyter": {
     "outputs_hidden": true
    },
    "tags": []
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "trigFired\n",
      "trigFired16\n",
      "trigFired17\n",
      "trigFired18\n",
      "eventNum\n",
      "lumiSec\n",
      "runNum\n",
      "METFiltersFailBits\n",
      "nElectron\n",
      "Electron_pt\n",
      "Electron_eta\n",
      "Electron_etaErr\n",
      "Electron_phi\n",
      "Electron_phiErr\n",
      "Electron_IDcutVeto\n",
      "Electron_IDcutLoose\n",
      "Electron_IDcutMed\n",
      "Electron_IDcutTight\n",
      "Electron_IDmvaIso90\n",
      "Electron_IDmvaIso80\n",
      "Electron_IDmvaIsoLoose\n",
      "Electron_IDmva90\n",
      "Electron_IDmva80\n",
      "Electron_IDmvaLoose\n",
      "Electron_angRes\n",
      "Electron_e\n",
      "Electron_vxy\n",
      "Electron_vz\n",
      "Electron_dxy\n",
      "Electron_dxyErr\n",
      "Electron_dz\n",
      "Electron_dzErr\n",
      "Electron_trkChi2\n",
      "Electron_trkIso\n",
      "Electron_trkRelIso\n",
      "Electron_calIso\n",
      "Electron_calRelIso\n",
      "Electron_PFIso4\n",
      "Electron_PFRelIso4\n",
      "Electron_PFIso3\n",
      "Electron_PFRelIso3\n",
      "Electron_PFIso8\n",
      "Electron_PFRelIso8\n",
      "Electron_PFIso\n",
      "Electron_PFRelIso\n",
      "Electron_trkProb\n",
      "Electron_numTrackerHits\n",
      "Electron_numPixHits\n",
      "Electron_numStripHits\n",
      "Electron_charge\n",
      "nLptElectron\n",
      "LptElectron_pt\n",
      "LptElectron_eta\n",
      "LptElectron_etaErr\n",
      "LptElectron_phi\n",
      "LptElectron_phiErr\n",
      "LptElectron_ID\n",
      "LptElectron_angRes\n",
      "LptElectron_e\n",
      "LptElectron_vxy\n",
      "LptElectron_vz\n",
      "LptElectron_dxy\n",
      "LptElectron_dxyErr\n",
      "LptElectron_dz\n",
      "LptElectron_dzErr\n",
      "LptElectron_trkChi2\n",
      "LptElectron_trkIso\n",
      "LptElectron_trkRelIso\n",
      "LptElectron_calIso\n",
      "LptElectron_calRelIso\n",
      "LptElectron_PFIso4\n",
      "LptElectron_PFRelIso4\n",
      "LptElectron_PFIso3\n",
      "LptElectron_PFRelIso3\n",
      "LptElectron_PFIso8\n",
      "LptElectron_PFIso\n",
      "LptElectron_PFRelIso\n",
      "LptElectron_PFRelIso8\n",
      "LptElectron_trkProb\n",
      "LptElectron_numTrackerHits\n",
      "LptElectron_numPixHits\n",
      "LptElectron_numStripHits\n",
      "LptElectron_charge\n",
      "LptElectron_minDRtoReg\n",
      "nPhoton\n",
      "Photon_et\n",
      "Photon_eta\n",
      "Photon_phi\n",
      "nootPhoton\n",
      "ootPhoton_et\n",
      "ootPhoton_eta\n",
      "ootPhoton_phi\n",
      "nConversion\n",
      "Conversion_pt\n",
      "Conversion_eta\n",
      "Conversion_phi\n",
      "Conversion_energy\n",
      "Conversion_px\n",
      "Conversion_py\n",
      "Conversion_pz\n",
      "Conversion_vxy\n",
      "Conversion_vz\n",
      "Conversion_x\n",
      "Conversion_y\n",
      "Conversion_z\n",
      "Conversion_trk1_innerPt\n",
      "Conversion_trk1_innerEta\n",
      "Conversion_trk1_innerPhi\n",
      "Conversion_trk1_outerPt\n",
      "Conversion_trk1_outerEta\n",
      "Conversion_trk1_outerPhi\n",
      "Conversion_trk2_innerPt\n",
      "Conversion_trk2_innerEta\n",
      "Conversion_trk2_innerPhi\n",
      "Conversion_trk2_outerPt\n",
      "Conversion_trk2_outerEta\n",
      "Conversion_trk2_outerPhi\n",
      "nPFJetAll\n",
      "nPFJet\n",
      "PFJet_pt\n",
      "PFJet_eta\n",
      "PFJet_phi\n",
      "PFJet_bTag\n",
      "PFJet_METdPhi\n",
      "PFJet_CHEF\n",
      "PFJet_NHEF\n",
      "PFJet_CEEF\n",
      "PFJet_NEEF\n",
      "PFJet_corrNumDaughters\n",
      "PFJet_corrCHM\n",
      "PFJet_corrJESUp_pt\n",
      "PFJet_corrJESUp_eta\n",
      "PFJet_corrJESUp_phi\n",
      "PFJet_corrJESDown_pt\n",
      "PFJet_corrJESDown_eta\n",
      "PFJet_corrJESDown_phi\n",
      "PFJet_corrJERUp_pt\n",
      "PFJet_corrJERUp_eta\n",
      "PFJet_corrJERUp_phi\n",
      "PFJet_corrJERDown_pt\n",
      "PFJet_corrJERDown_eta\n",
      "PFJet_corrJERDown_phi\n",
      "HEM_flag\n",
      "PFMET_ET\n",
      "PFMET_pt\n",
      "PFMET_phi\n",
      "PFMET_JESUpPt\n",
      "PFMET_JESUpPhi\n",
      "PFMET_JESDownPt\n",
      "PFMET_JESDownPhi\n",
      "PFMET_JERUpPt\n",
      "PFMET_JERUpPhi\n",
      "PFMET_JERDownPt\n",
      "PFMET_JERDownPhi\n",
      "PFMET_JetResUpSmearPt\n",
      "PFMET_JetResUpSmearPhi\n",
      "PFMET_JetResDownSmearPt\n",
      "PFMET_JetResDownSmearPhi\n",
      "CaloMET_ET\n",
      "CaloMET_pt\n",
      "CaloMET_phi\n",
      "rho\n",
      "nvtx\n",
      "vtx_typ\n",
      "vtx_vxy\n",
      "vtx_sigmavxy\n",
      "vtx_vx\n",
      "vtx_vy\n",
      "vtx_vz\n",
      "vtx_reduced_chi2\n",
      "vtx_prob\n",
      "vtx_dR\n",
      "vtx_sign\n",
      "vtx_minDxy\n",
      "vtx_METdPhi\n",
      "vtx_pt\n",
      "vtx_eta\n",
      "vtx_phi\n",
      "vtx_energy\n",
      "vtx_m\n",
      "vtx_px\n",
      "vtx_py\n",
      "vtx_pz\n",
      "vtx_PFIso4\n",
      "vtx_PFRelIso4\n",
      "vtx_PFIso3\n",
      "vtx_PFRelIso3\n",
      "vtx_PFIso8\n",
      "vtx_PFRelIso8\n",
      "vtx_e1_typ\n",
      "vtx_e1_idx\n",
      "vtx_e2_typ\n",
      "vtx_e2_idx\n",
      "genWgt\n",
      "genPU_obs\n",
      "genPU_true\n",
      "nGenJet\n",
      "GenJet_pt\n",
      "GenJet_eta\n",
      "GenJet_phi\n",
      "GenJet_METdPhi\n",
      "GenMET_pt\n",
      "GenMET_phi\n",
      "GenMET_px\n",
      "GenMET_py\n",
      "GenMET_ET\n",
      "nGenPart\n",
      "GenPart_ID\n",
      "GenPart_charge\n",
      "GenPart_pt\n",
      "GenPart_eta\n",
      "GenPart_phi\n",
      "GenPart_e\n",
      "GenPart_px\n",
      "GenPart_py\n",
      "GenPart_pz\n",
      "GenPart_vxy\n",
      "GenPart_vx\n",
      "GenPart_vy\n",
      "GenPart_vz\n",
      "GenPart_mass\n",
      "GenEle_charge\n",
      "GenEle_motherID\n",
      "GenEle_pt\n",
      "GenEle_eta\n",
      "GenEle_phi\n",
      "GenEle_energy\n",
      "GenEle_px\n",
      "GenEle_py\n",
      "GenEle_pz\n",
      "GenEle_vxy\n",
      "GenEle_vz\n",
      "GenEleClosest_typ\n",
      "GenEleClosest_ind\n",
      "GenEleClosest_dr\n",
      "GenEleClosestReg_ind\n",
      "GenEleClosestReg_dr\n",
      "GenEleClosestLpt_ind\n",
      "GenEleClosestLpt_dr\n",
      "GenPos_charge\n",
      "GenPos_motherID\n",
      "GenPos_pt\n",
      "GenPos_eta\n",
      "GenPos_phi\n",
      "GenPos_energy\n",
      "GenPos_px\n",
      "GenPos_py\n",
      "GenPos_pz\n",
      "GenPos_vxy\n",
      "GenPos_vz\n",
      "GenPosClosest_typ\n",
      "GenPosClosest_ind\n",
      "GenPosClosest_dr\n",
      "GenPosClosestReg_ind\n",
      "GenPosClosestReg_dr\n",
      "GenPosClosestLpt_ind\n",
      "GenPosClosestLpt_dr\n",
      "nGenEleTrkMatches\n",
      "genEleNearTk_pt\n",
      "genEleNearTk_eta\n",
      "genEleNearTk_phi\n",
      "genEleNearTk_dRgen\n",
      "genEleNearTk_pfIso3\n",
      "genEleNearTk_miniIso\n",
      "genEleNearTk_dxy\n",
      "genEleNearTk_dz\n",
      "genEleNearTk_highPurity\n",
      "genEleNearTk_loose\n",
      "genEleNearTk_charge\n",
      "genEleNearTk_numPixHits\n",
      "genEleNearTk_numStripHits\n",
      "genEleNearTk_fromPV\n",
      "genEleNearTk_tkIdx\n",
      "nGenPosTrkMatches\n",
      "genPosNearTk_pt\n",
      "genPosNearTk_eta\n",
      "genPosNearTk_phi\n",
      "genPosNearTk_dRgen\n",
      "genPosNearTk_pfIso3\n",
      "genPosNearTk_miniIso\n",
      "genPosNearTk_dxy\n",
      "genPosNearTk_dz\n",
      "genPosNearTk_highPurity\n",
      "genPosNearTk_loose\n",
      "genPosNearTk_charge\n",
      "genPosNearTk_numPixHits\n",
      "genPosNearTk_numStripHits\n",
      "genPosNearTk_fromPV\n",
      "genPosNearTk_tkIdx\n",
      "genEE_pt\n",
      "genEE_eta\n",
      "genEE_phi\n",
      "genEE_energy\n",
      "genEE_mass\n",
      "genEE_dr\n",
      "genEE_METdPhi\n"
     ]
    }
   ],
   "source": [
    "print(\"\\n\".join(t.keys()))"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "f4be55a3",
   "metadata": {},
   "source": [
    "There are a lot of entries here! But they're mostly grouped by name, so e.g. `Electron_*` are kinematic quantities for the electrons in each event, `LptElectron_*` are for the low-$p_T$ electrons, and so on.\n",
    "\n",
    "Let's look at the electrons as an example"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 20,
   "id": "dcd1b670-8d4a-4046-846e-2c0f52d3b1d4",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "[[], [], [], [], [4.74, 2.29]]\n"
     ]
    }
   ],
   "source": [
    "ele_pt = t['Electron_pt'].array()\n",
    "print(ele_pt[:5]) # print the electron pT for the first 5 events"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "2651e6db",
   "metadata": {},
   "source": [
    "The array `Electron_pt` is a *jagged array*, meaning each event has a variable number of entries (i.e. a variable number of electrons). Jagged arrrays are handled by the `awkward` package. \n",
    "\n",
    "Since this is a signal sample, we can also look at variables for the *generator-level* electron and positron in the iDM signal. These are accessed via the `GenEle_*` and `GenPos_*` branches"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 21,
   "id": "7060bb05",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "<Array [7.47, 4.14, 1.9, ... 10.2, 1.93, 6.49] type='30486 * float32'>"
      ]
     },
     "execution_count": 21,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "t[\"GenEle_pt\"].array()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 22,
   "id": "f244d6a6",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "<matplotlib.legend.Legend at 0x7f5232413b20>"
      ]
     },
     "execution_count": 22,
     "metadata": {},
     "output_type": "execute_result"
    },
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 800x600 with 1 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "# make a histogram of the gen electron and positron pT\n",
    "plt.figure(figsize=(8,6))\n",
    "bins = np.linspace(0,30,50)\n",
    "h1 = plt.hist(t['GenEle_pt'].array(),bins=bins,histtype='step',label='Gen Electrons')\n",
    "h1 = plt.hist(t['GenPos_pt'].array(),bins=bins,histtype='step',label='Gen Positrons')\n",
    "plt.xlabel(\"Lepton $p_T$ [GeV]\")\n",
    "plt.legend()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 23,
   "id": "cc414dff",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "Text(0, 0.5, 'Positron $p_T$ [GeV]')"
      ]
     },
     "execution_count": 23,
     "metadata": {},
     "output_type": "execute_result"
    },
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 800x600 with 1 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "# make a 2D histogram of the gen electron and positron pT\n",
    "plt.figure(figsize=(8,6))\n",
    "bins = np.linspace(0,15,50)\n",
    "h1 = plt.hist2d(t['GenEle_pt'].array(),t['GenPos_pt'].array(),bins=bins)\n",
    "plt.xlabel(\"Electon $p_T$ [GeV]\")\n",
    "plt.ylabel(\"Positron $p_T$ [GeV]\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "fdad5465",
   "metadata": {
    "tags": []
   },
   "source": [
    "## An easier way to do file I/O -- `NanoEvents`\n",
    "\n",
    "Uproot is nice for interacting with single files with a few branches, but gets a little cumbersome when there are many branches with long names. Instead, we can use the `coffea` package to interact with the files using the `NanoEvents` format. Here, I use the `loadNano` function from the `analysisTools` python file we imported at the top of tthe notebook. Go check out the source code if you want to know more about how this works!"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 8,
   "id": "3c4d1157",
   "metadata": {},
   "outputs": [],
   "source": [
    "inFile = fullList[0]\n",
    "events = tools.loadNano(inFile)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "335b5753",
   "metadata": {},
   "source": [
    "The `events` object now allows us to access event data in a structured way based on the branch names. For instance, all the `Electron` information can be accessed via `events.Electron` etc."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 9,
   "id": "11ae8b59",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "['genEleNearTk',\n",
       " 'PFJet',\n",
       " 'GenPosClosest',\n",
       " 'GenPosClosestReg',\n",
       " 'nGenEleTrkMatches',\n",
       " 'genPU',\n",
       " 'trigFired',\n",
       " 'GenMET',\n",
       " 'eventNum',\n",
       " 'runNum',\n",
       " 'vtx',\n",
       " 'genWgt',\n",
       " 'trigFired17',\n",
       " 'GenJet',\n",
       " 'GenPart',\n",
       " 'GenEleClosestLpt',\n",
       " 'CaloMET',\n",
       " 'lumiSec',\n",
       " 'GenEleClosest',\n",
       " 'GenPos',\n",
       " 'ootPhoton',\n",
       " 'GenEleClosestReg',\n",
       " 'nPFJetAll',\n",
       " 'HEM',\n",
       " 'Conversion',\n",
       " 'trigFired16',\n",
       " 'GenEle',\n",
       " 'genPosNearTk',\n",
       " 'trigFired18',\n",
       " 'METFiltersFailBits',\n",
       " 'rho',\n",
       " 'genEE',\n",
       " 'Electron',\n",
       " 'LptElectron',\n",
       " 'Photon',\n",
       " 'GenPosClosestLpt',\n",
       " 'nGenPosTrkMatches',\n",
       " 'PFMET']"
      ]
     },
     "execution_count": 9,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "events.fields"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 9,
   "id": "684e9b9d",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "<Array [[], [], [], ... 118, 8.45, 6.93, 2.46]] type='30486 * var * float32[para...'>"
      ]
     },
     "execution_count": 9,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "events.Electron.pt"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 10,
   "id": "a676f384",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "<Array [[], [2.1], [], ... [], [8.32, 3.27]] type='30486 * var * float32[paramet...'>"
      ]
     },
     "execution_count": 10,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "events.LptElectron.pt"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 11,
   "id": "d091a75b-d6cc-4a0a-9017-04a755716e61",
   "metadata": {},
   "outputs": [],
   "source": [
    "nJets = ak.count(events.PFJet.pt,axis=1)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 30,
   "id": "e5b9a99f-83e1-4511-9569-d7c8011d9fc8",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "<Array [1, 1, 1, 1, 1, 1, ... 1, 1, 1, 1, 1, 1] type='30486 * int64'>"
      ]
     },
     "execution_count": 30,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "nJets"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 10,
   "id": "d0ce7e60-518d-4975-b665-ba51ec9fbb0b",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "['genEleNearTk',\n",
       " 'PFJet',\n",
       " 'GenPosClosest',\n",
       " 'GenPosClosestReg',\n",
       " 'nGenEleTrkMatches',\n",
       " 'genPU',\n",
       " 'trigFired',\n",
       " 'GenMET',\n",
       " 'eventNum',\n",
       " 'runNum',\n",
       " 'vtx',\n",
       " 'genWgt',\n",
       " 'trigFired17',\n",
       " 'GenJet',\n",
       " 'GenPart',\n",
       " 'GenEleClosestLpt',\n",
       " 'CaloMET',\n",
       " 'lumiSec',\n",
       " 'GenEleClosest',\n",
       " 'GenPos',\n",
       " 'ootPhoton',\n",
       " 'GenEleClosestReg',\n",
       " 'nPFJetAll',\n",
       " 'HEM',\n",
       " 'Conversion',\n",
       " 'trigFired16',\n",
       " 'GenEle',\n",
       " 'genPosNearTk',\n",
       " 'trigFired18',\n",
       " 'METFiltersFailBits',\n",
       " 'rho',\n",
       " 'genEE',\n",
       " 'Electron',\n",
       " 'LptElectron',\n",
       " 'Photon',\n",
       " 'GenPosClosestLpt',\n",
       " 'nGenPosTrkMatches',\n",
       " 'PFMET']"
      ]
     },
     "execution_count": 10,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "events.fields"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 11,
   "id": "2f128e18-7e4e-4948-a2d0-52af11275d42",
   "metadata": {},
   "outputs": [],
   "source": [
    "# 1 or 2 jets in the event\n",
    "nJets = ak.count(events.PFJet.pt,axis=1)\n",
    "events = events[(nJets>0) & (nJets<3)]\n",
    "\n",
    "#################################\n",
    "## Calculating Additional Vars ##\n",
    "#################################\n",
    "routines.electronJetSeparation(events) # dR and dPhi between electrons and jets\n",
    "routines.electronIsoConePtSum(events) # for each electron, compute pT sum of any other electrons in event within dR < 0.3 of it\n",
    "routines.electronID(events) # electron kinematic/ID definition\n",
    "routines.vtxElectronConnection(events) # associate electrons to vertices\n",
    "routines.defineGoodVertices(events) # define \"good\" vertices based on whether associated electrons pass ID cuts\n",
    "\n",
    "#################################\n",
    "#### Demand >= 1 ee vertices ####\n",
    "#################################\n",
    "events.__setitem__(\"nGoodVtx\",ak.count(events.good_vtx.vxy,axis=1))\n",
    "events = events[events.nGoodVtx > 0]\n",
    "# define \"selected\" vertex based on selection criteria in the routine (nominally: lowest chi2)\n",
    "routines.selectBestVertex(events)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 15,
   "id": "8e9abc78-7aca-4423-9a61-d10ef4e345f6",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "<Array [1.66, -0.209, 0.0727, ... 1.23, 2] type='2557 * ?float32'>"
      ]
     },
     "execution_count": 15,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "events.sel_vtx.e1.IDscore[events.sel_vtx.e1_typ=='L']"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 14,
   "id": "ba2c5fc2-f6fb-4fbc-8fcb-eb458eda131d",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "<Array ['L', 'L', 'L', 'L', ... 'L', 'L', 'L'] type='2718 * option[string]'>"
      ]
     },
     "execution_count": 14,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "h.axes[0]\n",
    "selected = [s for s in slist if \"ctau-1\" in s]"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "93e86799",
   "metadata": {
    "tags": []
   },
   "source": [
    "## A basic analysis example\n",
    "Let's try applying a few basic cuts to the `events` object, which should give you a feel for how columnar analysis works\n",
    "\n",
    "First, we apply a cut demanding at least 150 GeV of missing energy:"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 33,
   "id": "50dede7f",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "<Array [False, False, False, ... True, False] type='30486 * bool'>"
      ]
     },
     "execution_count": 33,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "cut1 = events.PFMET.pt > 150\n",
    "cut1"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "2fdbef4f",
   "metadata": {},
   "source": [
    "As you can see, `cut1` is an array of booleans that tell us whether or not each event passes the missing energy cut. Let's apply this cut now"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 34,
   "id": "32652939",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Before cut : 30486 events\n",
      "After cut : 4751 events\n"
     ]
    }
   ],
   "source": [
    "print(f\"Before cut : {len(events)} events\")\n",
    "events = events[cut1]\n",
    "print(f\"After cut : {len(events)} events\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "994c7116",
   "metadata": {},
   "source": [
    "Now let's try filling a histogram. We use the [Hist](https://hist.readthedocs.io/en/latest/) package to manage our histograms"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 54,
   "id": "694d04ed",
   "metadata": {},
   "outputs": [
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "/uscms_data/d3/sbrightt/miniforge/envs/coffea2/lib/python3.8/site-packages/hist/__init__.py:58: UserWarning: Misspelling error, 'axes' should be 'axis'\n",
      "  warnings.warn(msg)\n"
     ]
    },
    {
     "data": {
      "text/html": [
       "<html>\n",
       "<div style=\"display:flex; align-items:center;\">\n",
       "<div style=\"width:290px;\">\n",
       "<svg xmlns=\"http://www.w3.org/2000/svg\" viewBox=\"-20 -270 290 290\">\n",
       "<text text-anchor=\"middle\" x=\"0\" y=\"13\" style=\"fill:currentColor;\">\n",
       "0\n",
       "</text>\n",
       "<text text-anchor=\"middle\" x=\"250\" y=\"13\" style=\"fill:currentColor;\">\n",
       "30\n",
       "</text>\n",
       "<text text-anchor=\"middle\" x=\"-10\" y=\"0\" style=\"fill:currentColor;\">\n",
       "0\n",
       "</text>\n",
       "<text text-anchor=\"middle\" x=\"-10\" y=\"-250\" style=\"fill:currentColor;\">\n",
       "2\n",
       "</text>\n",
       "<text text-anchor=\"middle\" x=\"125.0\" y=\"13\" style=\"fill:currentColor;\">\n",
       "pt\n",
       "</text>\n",
       "<text text-anchor=\"middle\" x=\"-10\" y=\"-125.0\" transform=\"rotate(-90,-10,-125.0)\" style=\"fill:currentColor;\">\n",
       "cat\n",
       "</text>\n",
       "<rect x=\"0.0\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.24147339699863576\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"8.333333333333334\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.11186903137789904\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"16.666666666666668\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.9713506139154161\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"25.0\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.7708049113233288\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"33.333333333333336\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.6180081855388814\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"41.666666666666664\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.5497953615279672\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"50.0\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.49113233287858116\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"58.333333333333336\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.407912687585266\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"66.66666666666667\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.31923601637107774\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"75.0\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.24283765347885403\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"83.33333333333333\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.20600272851296042\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"91.66666666666666\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.21145975443383355\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"100.0\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.14870395634379263\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"108.33333333333334\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.15688949522510232\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"116.66666666666667\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.11459754433833559\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"125.0\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.11323328785811732\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"133.33333333333334\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.11323328785811732\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"141.66666666666666\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.09822646657571624\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"150.0\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.07639836289222374\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"158.33333333333331\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.0723055934515689\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"166.66666666666666\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.058663028649386086\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"175.0\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.057298772169167796\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"183.33333333333331\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.03956343792633015\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"191.66666666666669\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.03956343792633015\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"200.0\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.04911323328785812\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"208.33333333333334\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.028649386084583898\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"216.66666666666669\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.02728512960436562\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"225.0\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.03819918144611187\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"233.33333333333334\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.04229195088676671\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"241.66666666666666\" y=\"-125.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.0354706684856753\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"0.0\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.0\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"8.333333333333334\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"1.0\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"16.666666666666668\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.9140518417462482\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"25.0\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.46521145975443384\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"33.333333333333336\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.27694406548431105\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"41.666666666666664\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.1923601637107776\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"50.0\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.11323328785811732\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"58.333333333333336\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.09276944065484312\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"66.66666666666667\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.07503410641200546\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"75.0\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.058663028649386086\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"83.33333333333333\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.04502046384720327\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"91.66666666666666\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.047748976807639835\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"100.0\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.0354706684856753\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"108.33333333333334\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.03137789904502046\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"116.66666666666667\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.025920873124147335\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"125.0\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.02728512960436562\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"133.33333333333334\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.021828103683492497\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"141.66666666666666\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.020463847203274217\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"150.0\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.02728512960436562\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"158.33333333333331\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.01364256480218281\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"166.66666666666666\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.019099590723055934\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"175.0\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.01773533424283765\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"183.33333333333331\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.01637107776261937\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"191.66666666666669\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.01637107776261937\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"200.0\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.01500682128240109\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"208.33333333333334\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.009549795361527967\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"216.66666666666669\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.01500682128240109\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"225.0\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.009549795361527967\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"233.33333333333334\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.01364256480218281\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "<rect x=\"241.66666666666666\" y=\"-250.0\" width=\"8.333\" height=\"125.0\" opacity=\"0.009549795361527967\" fill=\"currentColor\" stroke-width=\"0.1\"/>\n",
       "</svg>\n",
       "</div>\n",
       "<div style=\"flex=grow:1;\">\n",
       "Regular(30, 0, 30, name='pt')<br/>\n",
       "StrCategory(['Regular', 'Low $p_T$'], name='cat')<br/>\n",
       "<hr style=\"margin-top:.2em; margin-bottom:.2em;\"/>\n",
       "Double() Σ=7387.0 <em>(8434.0 with flow)</em>\n",
       "\n",
       "</div>\n",
       "</div>\n",
       "</html>"
      ],
      "text/plain": [
       "Hist(\n",
       "  Regular(30, 0, 30, name='pt'),\n",
       "  StrCategory(['Regular', 'Low $p_T$'], name='cat'),\n",
       "  storage=Double()) # Sum: 7387.0 (8434.0 with flow)"
      ]
     },
     "execution_count": 54,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "import hist\n",
    "h_pt = hist.Hist(\n",
    "    hist.axes.Regular(30,0,30,name=\"pt\"), # a \"Regular\" axis for storing the electron pT\n",
    "    hist.axes.StrCategory(['Regular','Low $p_T$'],name=\"cat\") # a \"string category\" axis for differentiating between low pT and regular electrons\n",
    ")\n",
    "# now fill the histogram\n",
    "h_pt.fill(\n",
    "    pt=ak.flatten(events.Electron.pt), # need to \"flatten\" the jagged array of Electron pT\n",
    "    cat=\"Regular\"\n",
    ")\n",
    "h_pt.fill(\n",
    "    pt=ak.flatten(events.LptElectron.pt),\n",
    "    cat='Low $p_T$'\n",
    ")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 59,
   "id": "3ff8e2dc",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "<matplotlib.legend.Legend at 0x7f520fc49700>"
      ]
     },
     "execution_count": 59,
     "metadata": {},
     "output_type": "execute_result"
    },
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 640x480 with 1 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "h_pt.plot1d(overlay='cat')\n",
    "plt.legend()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "c5383264-d8c6-4f31-aacc-681bdf0aa183",
   "metadata": {},
   "source": [
    "## Awkward arrays"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "febd2047-8341-4881-a6df-f8c90139a71b",
   "metadata": {},
   "source": [
    "Here's a fairly detailed example of the kinds of things you can do with `awkward` arrays (the default array type with coffea/uproot) and the `numba` package. Using `numba.njit()`, you can write accelerated \"event loop\"-style code. This programming style is a little more intuitive than pure columnar analysis, and easier to use for quick development. I think the more technically correct way to do this would be with `vectorize`, but this seems significantly more difficult to get right"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 6,
   "id": "faa211bd",
   "metadata": {},
   "outputs": [],
   "source": [
    "# load in NanoEvents\n",
    "inFile = fullList[0]\n",
    "events = tools.loadNano(inFile)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "90a5e8c9-5caa-48aa-8b3a-5a6cd923ada0",
   "metadata": {},
   "source": [
    "In this example, we'll write a for-loop algorithm that computes $\\Delta R = \\sqrt{\\Delta \\eta^2 + \\Delta \\phi^2}$ for every pair of electrons in each event. For technical reasons, the first argument of our jitted function needs to be an `ak.ArrayBuilder()` object to construct the output array"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 8,
   "id": "7784537c-8cd1-42c6-a5ed-6dcb3c85dd0f",
   "metadata": {},
   "outputs": [],
   "source": [
    "# helper function to compute deltaPhi correctly, ensuring the output is in the range [-pi,pi]\n",
    "# helper functions also need to have the njit decorator to be used inside *other* jitted functions\n",
    "@nb.njit()\n",
    "def deltaPhiSingle(phi1,phi2):\n",
    "    M_PI = 3.14159265358979323846264338328\n",
    "    dPhi = np.fmod(phi1-phi2,2*M_PI)\n",
    "    if dPhi < -1*M_PI:\n",
    "        dPhi += 2*M_PI\n",
    "    elif dPhi > M_PI:\n",
    "        dPhi -= 2*M_PI\n",
    "    return dPhi\n",
    "\n",
    "# helper for compute deltaR\n",
    "@nb.njit()\n",
    "def deltaRSingle(eta1,phi1,eta2,phi2):\n",
    "    return np.sqrt((eta1-eta2)**2 + deltaPhiSingle(phi1,phi2)**2)\n",
    "\n",
    "# define for-loop style function for pairwise electron dR computation with nb.njit() decorator\n",
    "@nb.njit()\n",
    "def elePairDR(b,e1_eta,e1_phi,e2_eta,e2_phi):\n",
    "    # e1_eta, e1_phi, etc. are Awkward arrays like [ [....], [....], ...., [...] ], \n",
    "    # so len(e1_eta) returns the *outer* length, i.e. the total number of events\n",
    "    nEvents = len(e1_eta) \n",
    "    for n in range(nEvents):\n",
    "        ne1 = len(e1_eta[n]) # number of electrons in reference collection, in this event\n",
    "        ne2 = len(e2_eta[n]) # number of electrons in target colletion (to compute dR w.r.t)\n",
    "        b.begin_list()\n",
    "        for i in range(ne1):\n",
    "            for j in range(ne2):\n",
    "                dphi = deltaPhiSingle(e1_phi[n][i],e2_phi[n][j])\n",
    "                dR = np.sqrt((e1_eta[n][i]-e2_eta[n][j])**2 + dphi**2)\n",
    "                b.real(dR)\n",
    "        b.end_list()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 9,
   "id": "4f2cc54a-076a-4ef7-98db-8204ab3de821",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "CPU times: user 1.58 s, sys: 46.8 ms, total: 1.63 s\n",
      "Wall time: 2.01 s\n"
     ]
    }
   ],
   "source": [
    "%%time\n",
    "b = ak.ArrayBuilder()\n",
    "reg_eles = events.Electron # normal electrons\n",
    "lpt_eles = events.LptElectron # low pT electrons\n",
    "elePairDR(b,reg_eles.eta,reg_eles.phi,lpt_eles.eta,lpt_eles.phi) #use jitted function to compute dR between regular and low pT electrons in each event\n",
    "output = b.snapshot()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 10,
   "id": "6b751fa9-f495-4d3c-ba4b-34bd4c8f66bc",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "<Array [[], [], ... 0.192, 5.52, 1.4, 4.08]] type='30486 * var * float64'>"
      ]
     },
     "execution_count": 10,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "output"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "432f446c-8247-423b-bf68-8aab818c5d9e",
   "metadata": {},
   "outputs": [],
   "source": []
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python [conda env:coffea2]",
   "language": "python",
   "name": "conda-env-coffea2-py"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.17"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}

NameError: name 'true' is not defined